# Verification Notebook
This notebook contains the verification tests performed for our MILP problem.

They follow from Sargeant 1998 criteria for verification and validation of simulation models.

Three types of tests are proposed:
1. Unit tests on the deterministic pre-processing.
2. Extreme/Degenerate condition tests.
3. Infeasibility tests.

In [ ]:
import numpy as np

## 1. Unit Tests (Deterministic Pre-Processing)

### &phi; - Rocket Equation

In [2]:
# Dummy Network
Connections = {0: [0,1], 1: [0,1]}      # 2 nodes
g_0 = 9.80665                           # m/s^2

delta_V = {0: {0: 0, 1: 2.0},                # ΔV [km/s]
           1: {0: 2.0, 1: 0}}

# Dummy Vehicle Set
I_sp    = np.array([0, 300])

In [ ]:
# Phi Function Definition - As in MILP code
def phi(i, j, v, dV=delta_V, I_sp=I_sp, g_0=g_0):
    if I_sp[v] == 0:
        return 1
    else:
        return 1 - np.exp(-(1000 * dV[i][j] / (I_sp[v] * g_0)))

In [ ]:
# TEST 1: Zero-Isp Special Case

test_1 = phi(0, 1, 0);              # node 0 to node 1; vehicle 0 (Isp = 0s)
expected_1 = 1

assert test_1 == expected_1, f"TEST 1 - FAIL: expected {expected_1}, got {test_1}"
print(f"TEST 1 - PASS: phi(Isp=0) = {test_1} (expected {expected_1})")

Test 1 - PASS: phi(Isp=0) = 1 (expected 1)


In [17]:
# TEST 2: Manual Rederivation of the Rocket Equation

i, j, v = 0, 1, 1                   # node 0 to node 1; vehicle 1 (Isp = 300s)
dV_ms = delta_V[i][j] * 1000        # km/s to m/s conversion
expected_2 = 1 - np.exp(-dV_ms / (I_sp[v] * g_0))

test_2 = phi(i, j, v)

assert test_2 == expected_2, f"TEST 2 - FAIL: expected {expected_2}, got {test_2}"
print(f"TEST 2 - PASS: phi = {test_2:.4f} (expected {expected_2:.4f})")

TEST 2 - PASS: phi = 0.4933 (expected 0.4933)


In [18]:
# TEST 3: Holdover Arc
# Staying at the same node should never burn propellant, for any vehicle

result_v0 = phi(0, 0, 0)            # Node 0 to Node 0; Vehicle 0 (Isp = 0s)
result_v1 = phi(0, 0, 1)            # Node 0 to Node 0, Vehicle 1 (Isp = 300s)

assert result_v0 == 1, f"TEST 3 - FAIL: expected 1 for vehicle 0 (Isp = 0s), got {result_v0}"
assert result_v1 == 0, f"TEST 3 - FAIL: expected 0 for vehicle 1 (Isp = 300s) and zero dV, got {result_v1}"

print(f"TEST 3 - PASS: phi(holdover, Isp=0s) = {result_v0}, phi(holdover, Isp=300s) = {result_v1}")

TEST 3 - PASS: phi(holdover, Isp=0s) = 1, phi(holdover, Isp=300s) = 0.0


### AllPossibleOutflowArcs